<a href="https://colab.research.google.com/github/Muhammad1917/Banking-agent/blob/main/The_Bank_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import pandas as pd

class BankingDataManager:

    def __init__(
        self,
        customers_path,
        orders_path,
        requests_path
    ):

        self.customers_path = customers_path
        self.orders_path = orders_path
        self.requests_path = requests_path

        self.customers = pd.read_excel(
            customers_path
        )

        self.orders = pd.read_excel(
            orders_path
        )
        self.CUSTOMER_COLUMNS = {
            "کد ملی": "national_id",
            "نام و نام خانوادگی": "name",
            "شماره تماس": "phone",
            "موجودی حساب": "balance",
            "امتیاز اعتباری": "credit_score",
            "تاریخ عضویت": "join_date",
            "تعداد تراکنش‌های ماه اخیر": "monthly_transactions",
        }

        self.ORDER_COLUMNS = {
            "کد ملی": "national_id",
            "نوع کارت": "card_type",
            "تاریخ سفارش": "order_date",
            "وضعیت": "status",
            "کد پیگیری": "tracking_code",
            "تاریخ تحویل تقریبی": "estimated_delivery",
            "آدرس ارسال": "shipping_address",
        }

        self.normalize()

    def normalize(self):
      self.customers = self.customers.rename(columns=self.CUSTOMER_COLUMNS)
      self.orders = self.orders.rename(columns=self.ORDER_COLUMNS)
      self.customers["national_id"] = (
        self.customers["national_id"]
        .astype(str)
        .str.strip()
      )
      self.orders["national_id"] = (
          self.orders["national_id"]
          .astype(str)
          .str.strip()
      )
      self.customers["national_id"] = (
          self.customers["national_id"]
          .str.replace(r"\.0$", "", regex=True)
          .str.zfill(10)
      )
      self.orders["tracking_code"] = (
          self.orders["tracking_code"]
          .astype(str)
          .str.replace(r"\.0$", "", regex=True)
          .str.strip()
      )

    def get_customer(self, national_id):

      """
      Return customer information by national ID.
      """

      national_id = str(national_id).strip().zfill(10)

      result = self.customers[
          customers["national_id"] == national_id
      ]

      if result.empty:
          return None

      return result.iloc[0].to_dict()

    def get_orders(self, national_id: str):
      """
      Return all card orders for a customer.
      """

      national_id = str(national_id).strip().zfill(10)
      print(national_id)
      result = self.orders[
          orders["national_id"] == national_id
      ]
      print(result)

      if result.empty:
          return []

      return result.to_dict(orient="records")

    def get_active_orders(self, national_id: str):
      """
      Return currently active card orders.
      """
      ACTIVE_STATUSES = {
        "در دست بررسی",
        "تأیید شده",
        "در حال چاپ",
        "ارسال به پست",
        "ارسال شده",
      }

      customer_orders = self.get_orders(national_id)
      print(customer_orders)
      return [
          order
          for order in customer_orders
          if order["status"] in ACTIVE_STATUSES
      ]


    def get_order_history(self,
          national_id: str,
          delivered_only: bool = False
      ):
      """
      Return customer's card order history.
      """

      customer_orders = self.get_orders(national_id)

      if delivered_only:
          customer_orders = [
              order
              for order in customer_orders
              if order["status"] == "تحویل داده شده"
          ]

      customer_orders.sort(
          key=lambda x: x["order_date"]
          if x["order_date"] is not None
          else pd.Timestamp.min,
          reverse=True
      )

      return customer_orders

    def create_request(
        self,
        national_id,
        request_type,
        description
    ):
      import os
      from datetime import datetime

      REQUEST_FILE = "/content/requests.xlsx"
      national_id = str(national_id).strip().zfill(10)

      # 1. Validate customer
      customer = self.get_customer(national_id)

      if customer is None:
          raise ValueError(
              "Customer with this national ID does not exist."
          )

      # 2. Get customer name
      name = customer["name"]

      # 3. Current date/time
      now = datetime.now()

      # 4. Load existing requests
      if os.path.exists(REQUEST_FILE):
          request_df = pd.read_excel(
              REQUEST_FILE
          )
      else:
          request_df = pd.DataFrame(
              columns=REQUEST_COLUMNS
          )

      # 5. Duplicate check
      if not request_df.empty:

        request_df["date"] = pd.to_datetime(
            request_df["date"],
            errors="coerce"
        )

        today = now.date()

        duplicate = request_df[
            (request_df["national_id"] == national_id)
            &
            (request_df["request_type"] == request_type)
            &
            (request_df["date"].dt.date == today)
        ]

        if not duplicate.empty:
            return {
                "success": False,
                "duplicate": True,
                "message": (
                    "این درخواست امروز قبلاً ثبت شده است."
                )
            }

        # 6. Create new request
        new_request = {
            "national_id": national_id,
            "name": name,
            "request_type": request_type,
            "description": description,
            "date": now,
            "status": "ثبت شده",
        }

        request_df = pd.concat(
            [
                request_df,
                pd.DataFrame([new_request])
            ],
            ignore_index=True
        )

        # 7. Save
        request_df.to_excel(
            REQUEST_FILE,
            index=False
        )

        return {
            "success": True,
            "duplicate": False,
            "request": new_request
        }

In [1]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    langchain-openai \
    langsmith

In [3]:
!pip install -q pandas openpyxl python-docx sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 60.4 MB/s eta 0:00:00


In [4]:
from docx import Document


def load_policy_document(path):
    doc = Document(path)

    parts = []

    for paragraph in doc.paragraphs:
        text = paragraph.text.strip()

        if text:
            parts.append(text)

    for table in doc.tables:
        for row in table.rows:
            cells = [
                cell.text.strip()
                for cell in row.cells
            ]

            if any(cells):
                parts.append(" | ".join(cells))

    return "\n".join(parts)

In [33]:
!ls


'card_orders(3) (1).xlsx'   drive
'card_orders(3).xlsx'	    sample_data
'customers (1).xlsx'	   'قوانین و مقررات بانک دایا_تدبیر (1).docx'
 customers.xlsx		   'قوانین و مقررات بانک دایا_تدبیر.docx'


In [34]:
policy_text = load_policy_document(
    "قوانین و مقررات بانک دایا_تدبیر.docx"
)

print(policy_text)

قوانین و مقررات بانک دایا‌تدبیر
نسخه 2.1 | آذر 1404
1. انواع حساب‌های بانکی
1.1 حساب جاری
موجودی اولیه: حداقل 500,000 تومان
کارمزد ماهانه: رایگان
تعداد تراکنش رایگان: نامحدود
امکان چک: دارد
سود: ندارد
مناسب برای: کسب‌وکارها و پرداخت‌های روزانه
1.2 حساب پس‌انداز کوتاه‌مدت
موجودی اولیه: حداقل 1,000,000 تومان
نرخ سود سالانه: 18%
برداشت: بدون محدودیت
کارمزد: رایگان
مناسب برای: پس‌انداز روزمره
1.3 حساب پس‌انداز بلندمدت
موجودی اولیه: حداقل 5,000,000 تومان
نرخ سود سالانه: 22%
مدت زمان: حداقل 6 ماه
جریمه برداشت زودهنگام: 3% از سود
مناسب برای: پس‌انداز بلندمدت و سرمایه‌گذاری
1.4 حساب سرمایه‌گذاری ویژه
موجودی اولیه: حداقل 50,000,000 تومان
نرخ سود سالانه: 25%
مدت زمان: حداقل 12 ماه
مشاوره رایگان: دارد
مدیر حساب اختصاصی: دارد
مناسب برای: سرمایه‌گذاران حرفه‌ای
2. انواع کارت‌های بانکی
2.1 کارت نقدی استاندارد
ویژگی‌های کلی:
کارمزد صدور: 200,000 تومان
کارمزد سالانه: 150,000 تومان
زمان صدور: 5-7 روز کاری
اعتبار: 4 سال
محدودیت‌های برداشت:
حداکثر برداشت روزانه از دستگاه خودپرداز: 20,000,000 تومان
حداکثر 

In [27]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/colab_lib')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}"')


Saving card_orders(3).xlsx to card_orders(3) (1).xlsx
Saving customers.xlsx to customers (1).xlsx
Saving قوانین و مقررات بانک دایا_تدبیر.docx to قوانین و مقررات بانک دایا_تدبیر (1).docx
User uploaded file "card_orders(3) (1).xlsx"
User uploaded file "customers (1).xlsx"
User uploaded file "قوانین و مقررات بانک دایا_تدبیر (1).docx"


In [30]:
!ls


'card_orders(3) (1).xlsx'   drive
'card_orders(3).xlsx'	    sample_data
'customers (1).xlsx'	   'قوانین و مقررات بانک دایا_تدبیر (1).docx'
 customers.xlsx		   'قوانین و مقررات بانک دایا_تدبیر.docx'


In [ ]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "YOUR_LANGSMITH_API_KEY"
os.environ["LANGCHAIN_PROJECT"] = "banking-agent"

In [39]:

db = BankingDataManager(
    customers_path="/content/customers.xlsx",
    orders_path="/content/card_orders(3).xlsx",
    requests_path="/content/requests.xlsx"
)

In [40]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
)

chunks = splitter.create_documents(
    [policy_text]
)

In [41]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={
        "device": "cpu"
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [42]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

/tmp/ipykernel_3349/3106790601.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [43]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)

In [44]:
docs = retriever.invoke(
    "شرایط دریافت وام چیست؟"
)

Langchain Tools

In [46]:
from langchain_core.tools import tool


In [47]:
@tool
def authenticate_customer(national_id: str) -> dict:
    """
    Authenticate a customer using their national ID.

    Use this before accessing customer-specific information.
    """

    customer = db.get_customer(national_id)

    if customer is None:
        return {
            "authenticated": False,
            "message": "مشتری با این کد ملی در سیستم پیدا نشد."
        }

    return {
        "authenticated": True,
        "national_id": customer["national_id"],
        "name": customer["name"]
    }

In [59]:


@tool

def search_bank_policy(query: str) -> str:
    """
    Search the bank policy document for information
    relevant to a user's policy question.
    """

    docs = retriever.invoke(query)

    if not docs:
        return "No relevant policy information was found."

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [48]:
@tool
def get_customer(national_id: str) -> dict:
    """
    Get customer information such as balance and credit score.

    Use this when a question requires customer-specific financial
    information.
    """

    customer = db.get_customer(national_id)

    if customer is None:
        return {
            "found": False,
            "message": "مشتری پیدا نشد."
        }

    return {
        "found": True,
        "customer": customer
    }

In [49]:
@tool
def get_orders(national_id: str) -> list:
    """
    Get all card orders for a customer.
    """

    return db.get_orders(national_id)

In [50]:
@tool
def get_active_orders(national_id: str) -> dict:
    """
    Get all currently active card orders for a customer.

    Use this for questions about:
    - card status
    - where the card is
    - whether the card has arrived
    - when the card will arrive
    """

    orders = db.get_active_orders(national_id)

    return {
        "count": len(orders),
        "orders": orders
    }

In [51]:
@tool
def get_order_history(national_id: str) -> dict:
    """
    Get the customer's card order history.

    Use this when the user asks about:
    - previous cards
    - last delivered card
    - historical orders
    """

    orders = db.get_order_history(national_id)

    return {
        "count": len(orders),
        "orders": orders
    }

In [52]:
@tool
def create_request(
    national_id: str,
    request_type: str,
    description: str
) -> dict:
    """
    Create a customer request.

    Request types:
    - وام
    - افتتاح حساب
    - سفارش کارت جدید
    - شکایت
    - استعلام
    - سایر
    """

    return db.create_request(
        national_id=national_id,
        request_type=request_type,
        description=description
    )

In [53]:
@tool
def calculate_order_age(order_date: str) -> dict:
    """
    Calculate how many calendar days have passed since
    a card order date.
    """

    order_date = pd.to_datetime(
        order_date,
        errors="coerce"
    )

    if pd.isna(order_date):
        return {
            "success": False,
            "message": "تاریخ سفارش نامعتبر است."
        }

    today = pd.Timestamp.now().normalize()

    days = (today - order_date.normalize()).days

    return {
        "success": True,
        "days_elapsed": days
    }

giving the model the tools

In [55]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [56]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
)

llm = HuggingFacePipeline(
    pipeline=pipe
)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [57]:
SYSTEM_PROMPT = """
تو یک Agent بانکی هوشمند هستی که به زبان فارسی با مشتری صحبت می‌کنی.

وظایف اصلی:
- شناسایی مشتری با کد ملی
- پیگیری سفارش کارت
- پاسخ به سوالات قوانین و سیاست‌های بانک
- پاسخ به سوالات ترکیبی با استفاده از چند منبع
- ثبت درخواست مشتری

قوانین مهم:

1. برای اطلاعات شخصی مشتری هرگز حدس نزن.
   اطلاعات مشتری فقط باید از ابزارهای مربوط به customer دریافت شود.

2. برای اطلاعات سفارش کارت از ابزارهای سفارش کارت استفاده کن.

3. برای قوانین بانک، شرایط وام، انواع کارت، زمان صدور کارت،
   محدودیت برداشت و سایر policyها از search_bank_policy استفاده کن.

4. اگر سوال نیاز به چند منبع دارد، تمام منابع لازم را بررسی کن.
   مثلا برای بررسی امکان دریافت وام:
   - اطلاعات مشتری
   - policy وام
   هر دو لازم هستند.

5. اگر کاربر چند سفارش کارت فعال دارد، همه سفارش‌ها را بررسی کن
   و اگر لازم بود از کاربر بپرسد کدام کارت را منظور دارد.

6. اگر سفارش فعال وجود ندارد، این موضوع را واضح به کاربر بگو.

7. اگر سوال به سفارش قبلی و تحویل‌شده مربوط است،
   از تاریخچه سفارش‌ها استفاده کن.

8. برای محاسبات تاریخ و تعداد روزها به نتیجه ابزارها اعتماد کن
   و تاریخ را خودت حدس نزن.

9. قبل از ثبت درخواست، هویت مشتری را بررسی کن.

10. هنگام ثبت درخواست، نوع درخواست و توضیحات مناسب را مشخص کن.

11. اگر اطلاعات لازم برای پاسخ وجود ندارد، سوال روشن‌کننده بپرس.

12. پاسخ نهایی را کوتاه، واضح و به زبان فارسی ارائه کن.

13. اطلاعاتی مثل کد پیگیری، موجودی، امتیاز اعتباری،
    آدرس و تاریخ را فقط از داده واقعی ابزارها نمایش بده.

14. هرگز به کاربر نگو که اطلاعاتی را از خودت حدس زده‌ای.

انواع درخواست:
- وام
- افتتاح حساب
- سفارش کارت جدید
- شکایت
- استعلام
- سایر
"""

In [60]:
tools = [
    authenticate_customer,
    get_customer,
    get_orders,
    get_active_orders,
    get_order_history,
    search_bank_policy,
    create_request,
]

In [ ]:
state = {
    "national_id": None,
    "last_intent": None,
    "last_entity": None,
    "conversation": []
}

In [64]:
!pip show langchain langchain-core langchain-community

Name: langchain
Version: 1.4.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-core
Version: 1.6.6
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: httpx, jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-classic, langchain-community, langchain-huggingface, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk
---
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
Li

In [65]:
from langchain.agents import create_agent

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)


In [69]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),

    MessagesPlaceholder(
        variable_name="chat_history"
    ),

    ("human", "{input}"),

    MessagesPlaceholder(
        variable_name="agent_scratchpad"
    )
])

In [72]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_history = InMemoryChatMessageHistory()

/tmp/ipykernel_3349/2835580824.py:3: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  chat_history = InMemoryChatMessageHistory()


In [84]:
from langchain_huggingface import ChatHuggingFace
chat_model = ChatHuggingFace(
    llm=llm
)

In [85]:
agent = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

In [86]:
def chat(user_message):

    chat_history.add_user_message(user_message)

    result = agent.invoke({
        "messages": chat_history.messages
    })

    answer = result["messages"][-1].content

    chat_history.add_ai_message(answer)

    return answer
print(chat('سلام، کد ملی من 0123456789 است. کارتم چی شد؟"'))

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [77]:
def chat(user_message):

    chat_history.add_user_message(
        user_message
    )

    result = agent.invoke({
        "input": user_message,
        "chat_history": chat_history.messages
    })

    answer = result["output"]

    chat_history.add_ai_message(
        answer
    )

    return answer



AttributeError: 'HuggingFacePipeline' object has no attribute 'bind_tools'

In [ ]:
response = agent.invoke({
    "input": "سلام، کد ملی من 0123456789 است. کارتم چی شد؟"
})

print(response["output"])

In [ ]:
MessagesPlaceholder(
    variable_name="chat_history"
)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),

    MessagesPlaceholder(
        variable_name="chat_history"
    ),

    ("human", "{input}"),

    MessagesPlaceholder(
        variable_name="agent_scratchpad"
    )
])